# CLASIFICADOR DE CORREO ELECTRONICO SPAM UTILIZANDO FINE TUNING A MODELO PREENTRENADO

In [3]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `codigog4` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `codigog4`

# 🛠️ Paso 1: Instalar librerías necesarias

In [1]:
!pip install datasets==3.5.0 transformers==4.48.3 evaluate==0.4.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 514.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 13.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled transformers-4.53.3
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are install

# 📚 Paso 2: Cargar un dataset

In [4]:
from datasets import load_dataset

ds = load_dataset("ucirvine/sms_spam")

In [5]:
ds

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 5574
    })
})

# ✅ Paso 3: Preparar los datos

## DIVIDEMOS DATASET EN TRAIN Y TEST

In [8]:
ds = ds["train"].train_test_split(test_size=0.2)
ds

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 3567
    })
    test: Dataset({
        features: ['sms', 'label'],
        num_rows: 892
    })
})

## TOKENIZAMOS LA DATA

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["sms"], padding="max_length", truncation=True)


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [10]:
ds_train = ds['train']
ds_test = ds['test']
tokenized_train = ds_train.map(tokenize_function, batched=True)
tokenized_test = ds_test.map(tokenize_function, batched=True)

Map:   0%|          | 0/3567 [00:00<?, ? examples/s]

Map:   0%|          | 0/892 [00:00<?, ? examples/s]

## PREPARAMOS DATA LOADERS

In [11]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# 🚀 Paso 4: Cargar el modelo para clasificación

In [12]:
from transformers import AutoModelForSequenceClassification

# Cargamos BERT con una capa final para clasificación binaria (2 clases)
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# ⚙️ Paso 5: Configurar los parámetros de entrenamiento

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",             # 📁 Carpeta donde se guardarán los resultados del modelo entrenado.
    evaluation_strategy="epoch",        # 📊 Estrategia de evaluación: aquí se evalúa el modelo al final de cada época.
    save_strategy="epoch",              # 💾 Guarda el modelo también al final de cada época.
    logging_dir="./logs",               # 📝 Directorio donde se almacenan los registros del entrenamiento (logs).
    per_device_train_batch_size=8,      # 🧠 Tamaño del batch (lote) por dispositivo para entrenamiento. Aquí se usan 8 ejemplos por lote.
    per_device_eval_batch_size=8,       # 🧠 Tamaño del batch por dispositivo para evaluación.
    num_train_epochs=3,                 # 🔁 Número total de épocas de entrenamiento (pasa 3 veces por todos los datos).
    weight_decay=0.01,                  # ⚖️ Aplicación de regularización L2 (weight decay) para evitar overfitting.
    logging_steps=10,                   # 📌 Número de pasos de entrenamiento entre cada log (registro en consola).
    load_best_model_at_end=True,        # 🏆 Carga automáticamente el mejor modelo evaluado al final del entrenamiento.
    save_total_limit=2                  # 🧹 Limita a 2 el número total de checkpoints guardados para ahorrar espacio.
)

# 🧪 Paso 6: Definir el evaluador (función de métricas)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, predictions)
    prec, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    return {"accuracy": acc, "precision": prec, "recall": recall, "f1": f1}

# 🏋️ Paso 7: Entrenar el modelo con Trainer

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Iniciar entrenamiento
trainer.train()

# 📈 Paso 8: Evaluar el modelo

In [ ]:
trainer.evaluate()

# PROBAR CON UN CORREO NUEVO SI ES ES SPAM O NO(USAR CHATGPT O GEMINI)